# Pet-mischief-detection

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# INSTALL DEPENDENCIES (run this cell first in Colab)
# ═══════════════════════════════════════════════════════════════════════════════
!pip install -q ultralytics
!pip install -q opencv-python-headless

print("✓ Dependencies installed!")


In [1]:
import platform
import os

print(f"OS: {platform.platform()}")
print(f"CPU Count: {os.cpu_count()}")
!nvidia-smi

OS: Linux-6.6.113+-x86_64-with-glibc2.35
CPU Count: 2
Tue Apr 28 08:21:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  

## Global Configuration

Shared parameters used across all tasks. **Run this cell first.**

In [2]:
"""
Global Configuration
====================
Shared parameters for the entire pipeline.
"""
from pathlib import Path

# ═══════════════════════════════════════════════════════════════════════════════
# DATASET CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════════

# Target classes for pet mischief detection
TARGET_CLASSES = {
    "cat": "cat",
    "cup": "cup",
    "laptop": "laptop",
    "keyboard": "keyboard",
    "vase": "vase",
    "potted plant": "plant",
    "scissors": "scissors",
}

# True = any image with target class (more data); False = cat + other required
ANY_TARGET_CLASS = True

# ═══════════════════════════════════════════════════════════════════════════════
# DIRECTORY PATHS
# ═══════════════════════════════════════════════════════════════════════════════

ANNOTATIONS_DIR = Path("data/annotations")
COCO_FILTERED_DIR = Path("coco_filtered")
CURATED_YOLO_DIR = Path("curated_yolo")
TRAINING_DATA_YAML = CURATED_YOLO_DIR / "data.yaml"

# ═══════════════════════════════════════════════════════════════════════════════
# REPRODUCIBILITY & SPLITS
# ═══════════════════════════════════════════════════════════════════════════════

SEED = 42
TRAIN_PCT, VAL_PCT, TEST_PCT = 80, 10, 10

# ═══════════════════════════════════════════════════════════════════════════════
# DOWNLOAD SETTINGS
# ═══════════════════════════════════════════════════════════════════════════════

COCO_ANNOTATIONS_URL = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"
COCO_IMAGE_BASE_URL = "http://images.cocodataset.org"
DOWNLOAD_WORKERS = 8
DOWNLOAD_RETRIES = 3

# ═══════════════════════════════════════════════════════════════════════════════
# VISUALIZATION
# ═══════════════════════════════════════════════════════════════════════════════

GRAPH_FILENAMES = ["results.png", "confusion_matrix.png", "BoxF1_curve.png", "BoxR_curve.png", "BoxP_curve.png"]

# Print summary
print("=" * 60)
print("GLOBAL CONFIGURATION LOADED")
print("=" * 60)
print(f"Classes ({len(TARGET_CLASSES)}): {list(TARGET_CLASSES.values())}")
print(f"Filter: {'Any target class' if ANY_TARGET_CLASS else 'Cat + other required'}")
print(f"Split: {TRAIN_PCT}/{VAL_PCT}/{TEST_PCT}")
print(f"Seed: {SEED}")
print(f"Training data: {TRAINING_DATA_YAML}")


GLOBAL CONFIGURATION LOADED
Classes (7): ['cat', 'cup', 'laptop', 'keyboard', 'vase', 'plant', 'scissors']
Filter: Any target class
Split: 80/10/10
Seed: 42
Training data: curated_yolo/data.yaml


## Task 1: COCO Subset Preparation

Downloads COCO annotations, filters classes, downloads images, converts to YOLO format.

**Output:** `coco_filtered/`

In [3]:
"""
Task 1: COCO Subset Preparation
Uses global config: TARGET_CLASSES, ANY_TARGET_CLASS, ANNOTATIONS_DIR, COCO_FILTERED_DIR
"""
import json
import sys
import urllib.error
import urllib.request
import zipfile
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

def _download(url, dest, desc=""):
    dest.parent.mkdir(parents=True, exist_ok=True)
    print(f"Downloading {desc or url}")
    urllib.request.urlretrieve(url, dest)

def ensure_annotations():
    train_json = ANNOTATIONS_DIR / "instances_train2017.json"
    if train_json.is_file():
        print(f"✓ Annotations exist at {ANNOTATIONS_DIR}")
        return ANNOTATIONS_DIR
    
    extract_parent = ANNOTATIONS_DIR.parent
    extract_parent.mkdir(parents=True, exist_ok=True)
    zip_path = extract_parent / "annotations_trainval2017.zip"
    if not zip_path.is_file():
        _download(COCO_ANNOTATIONS_URL, zip_path, "COCO annotations")
    
    print(f"Extracting to {extract_parent}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_parent)
    return ANNOTATIONS_DIR

def filter_split(split_name, ann_path, img_subdir, output_dir):
    with ann_path.open() as f:
        coco = json.load(f)
    
    target_cats = {c["id"]: c["name"] for c in coco["categories"] if c["name"] in TARGET_CLASSES}
    id_remap = {old: new for new, old in enumerate(sorted(target_cats.keys()))}
    export_names = [TARGET_CLASSES[target_cats[oid]] for oid in sorted(target_cats.keys())]
    
    if ANY_TARGET_CLASS:
        kept_ids = {a["image_id"] for a in coco["annotations"] if a["category_id"] in target_cats}
    else:
        cat_id = [cid for cid, name in target_cats.items() if name == "cat"][0]
        other_ids = {cid for cid in target_cats if cid != cat_id}
        by_image = defaultdict(set)
        for a in coco["annotations"]:
            if a["category_id"] in target_cats:
                by_image[a["image_id"]].add(a["category_id"])
        kept_ids = {img_id for img_id, cats in by_image.items() if cat_id in cats and cats & other_ids}
    
    filtered_anns = [a for a in coco["annotations"] if a["category_id"] in target_cats and a["image_id"] in kept_ids]
    filtered_images = [img for img in coco["images"] if img["id"] in kept_ids]
    
    for ann in filtered_anns:
        ann["category_id"] = id_remap[ann["category_id"]]
    
    out_ann_dir = output_dir / "annotations"
    out_ann_dir.mkdir(parents=True, exist_ok=True)
    new_cats = [{"id": new, "name": TARGET_CLASSES[target_cats[old]], "supercategory": "object"} for old, new in id_remap.items()]
    with (out_ann_dir / f"instances_{split_name}.json").open("w") as f:
        json.dump({"categories": new_cats, "images": filtered_images, "annotations": filtered_anns}, f)
    
    print(f"[{split_name}] {len(filtered_images)} images, {len(filtered_anns)} annotations")
    return filtered_images, export_names

def download_images(split_name, img_subdir, filtered_images, output_dir):
    out_dir = output_dir / "images" / split_name
    out_dir.mkdir(parents=True, exist_ok=True)
    base = f"{COCO_IMAGE_BASE_URL}/{img_subdir}"
    
    def fetch(url, dest):
        if dest.is_file(): return True
        try:
            urllib.request.urlretrieve(url, dest)
            return True
        except: return False
    
    tasks = [(f"{base}/{img['file_name']}", out_dir / img["file_name"]) for img in filtered_images]
    print(f"[{split_name}] Downloading {len(tasks)} images...")
    
    done = 0
    with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as ex:
        for ok in ex.map(lambda t: fetch(*t), tasks):
            done += 1
            if done % 500 == 0: print(f"  {done}/{len(tasks)}")
    print(f"[{split_name}] ✓ Done")

def coco_to_yolo(ann_json, out_label_dir):
    with ann_json.open() as f:
        data = json.load(f)
    img_info = {img["id"]: img for img in data["images"]}
    ann_by_img = defaultdict(list)
    for ann in data["annotations"]:
        ann_by_img[ann["image_id"]].append(ann)
    
    out_label_dir.mkdir(parents=True, exist_ok=True)
    for img_id, anns in ann_by_img.items():
        meta = img_info[img_id]
        w, h = meta["width"], meta["height"]
        lines = [f"{a['category_id']} {(a['bbox'][0]+a['bbox'][2]/2)/w:.6f} {(a['bbox'][1]+a['bbox'][3]/2)/h:.6f} {a['bbox'][2]/w:.6f} {a['bbox'][3]/h:.6f}" for a in anns]
        (out_label_dir / f"{Path(meta['file_name']).stem}.txt").write_text("\n".join(lines))

def write_data_yaml(output_dir, names):
    lines = [f"path: {output_dir}", "train: images/train", "val: images/val", "", f"nc: {len(names)}", "names:"]
    lines += [f"  {i}: {n}" for i, n in enumerate(names)]
    (output_dir / "data.yaml").write_text("\n".join(lines))

SPLITS_CFG = {"train": ("instances_train2017.json", "train2017"), "val": ("instances_val2017.json", "val2017")}

def run_task1(skip_download=False):
    print("=" * 60)
    print("TASK 1: COCO SUBSET PREPARATION")
    print("=" * 60)
    
    ann_root = ensure_annotations()
    COCO_FILTERED_DIR.mkdir(parents=True, exist_ok=True)
    
    names = None
    total = 0
    for split, (ann_file, img_subdir) in SPLITS_CFG.items():
        imgs, names = filter_split(split, ann_root / ann_file, img_subdir, COCO_FILTERED_DIR)
        total += len(imgs)
        if not skip_download:
            download_images(split, img_subdir, imgs, COCO_FILTERED_DIR)
        coco_to_yolo(COCO_FILTERED_DIR / "annotations" / f"instances_{split}.json", COCO_FILTERED_DIR / "labels" / split)
    
    write_data_yaml(COCO_FILTERED_DIR, names)
    print(f"\n✓ Task 1 complete: {total} images -> {COCO_FILTERED_DIR}/")
    return names


In [ ]:
# Uncomment to run:
run_task1(skip_download=False)

TASK 1: COCO SUBSET PREPARATION
Extracting to data
[train] 21673 images, 49989 annotations
[train] Downloading 21673 images...
  500/21673


## Task 2: Dataset Splitting

Creates 80/10/10 train/val/test split.

**Input:** `coco_filtered/` | **Output:** `curated_yolo/`

In [ ]:
"""
Task 2: Dataset Splitting
Uses global config: COCO_FILTERED_DIR, CURATED_YOLO_DIR, SEED, TRAIN_PCT, VAL_PCT, TEST_PCT
"""
import json
import random
import re
import shutil
from collections import Counter
from datetime import UTC, datetime

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

def parse_names(yaml_path):
    text = yaml_path.read_text()
    names = {}
    in_names = False
    for line in text.splitlines():
        if line.strip().startswith("names:"):
            in_names = True
            continue
        if in_names:
            m = re.match(r"\s*(\d+)\s*:\s*(\S+)", line)
            if m: names[int(m.group(1))] = m.group(2)
            elif line.strip() and not line.startswith(" "): break
    return [names[i] for i in sorted(names)]

def collect_pairs(source):
    pairs = []
    for sub in ("train", "val"):
        img_root, lbl_root = source / "images" / sub, source / "labels" / sub
        if not img_root.is_dir(): continue
        for img in sorted(img_root.iterdir()):
            if img.suffix.lower() in IMAGE_EXTS:
                lbl = lbl_root / f"{img.stem}.txt"
                if lbl.is_file(): pairs.append((img, lbl))
    return pairs

def count_instances(label_paths):
    c = Counter()
    for p in label_paths:
        for line in p.read_text().splitlines():
            if line.strip(): c[int(line.split()[0])] += 1
    return c

def run_task2():
    print("=" * 60)
    print("TASK 2: DATASET SPLITTING")
    print("=" * 60)
    print(f"Input: {COCO_FILTERED_DIR}/ | Output: {CURATED_YOLO_DIR}/")
    print(f"Split: {TRAIN_PCT}/{VAL_PCT}/{TEST_PCT} | Seed: {SEED}")
    
    if not (COCO_FILTERED_DIR / "data.yaml").exists():
        raise FileNotFoundError("Run Task 1 first!")
    
    class_names = parse_names(COCO_FILTERED_DIR / "data.yaml")
    pairs = collect_pairs(COCO_FILTERED_DIR)
    n = len(pairs)
    
    idx = list(range(n))
    random.Random(SEED).shuffle(idx)
    n_train, n_val = (n * TRAIN_PCT) // 100, (n * VAL_PCT) // 100
    splits = {"train": idx[:n_train], "val": idx[n_train:n_train+n_val], "test": idx[n_train+n_val:]}
    
    print(f"Total pairs: {n} -> train={len(splits['train'])}, val={len(splits['val'])}, test={len(splits['test'])}")
    
    shutil.rmtree(CURATED_YOLO_DIR / "images", ignore_errors=True)
    shutil.rmtree(CURATED_YOLO_DIR / "labels", ignore_errors=True)
    
    all_stats = {}
    for split, indices in splits.items():
        (CURATED_YOLO_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
        (CURATED_YOLO_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)
        lbls = []
        for i in indices:
            src_img, src_lbl = pairs[i]
            shutil.copy2(src_img, CURATED_YOLO_DIR / "images" / split / src_img.name)
            shutil.copy2(src_lbl, CURATED_YOLO_DIR / "labels" / split / src_lbl.name)
            lbls.append(CURATED_YOLO_DIR / "labels" / split / src_lbl.name)
        inst = count_instances(lbls)
        all_stats[split] = {"images": len(indices), "instances": {class_names[k]: inst.get(k,0) for k in range(len(class_names))}}
    
    # Write data.yaml
    lines = [f"path: {CURATED_YOLO_DIR}", "train: images/train", "val: images/val", "test: images/test", "", f"nc: {len(class_names)}", "names:"]
    lines += [f"  {i}: {n}" for i, n in enumerate(class_names)]
    (CURATED_YOLO_DIR / "data.yaml").write_text("\n".join(lines))
    
    # Write manifest
    manifest = {"seed": SEED, "splits": all_stats, "class_names": class_names}
    (CURATED_YOLO_DIR / "dataset_manifest.json").write_text(json.dumps(manifest, indent=2))
    
    print("\n" + "=" * 60)
    print("TASK 2 COMPLETE")
    print("=" * 60)
    for split, stats in all_stats.items():
        print(f"\n[{split.upper()}] {stats['images']} images")
        for cls, cnt in stats["instances"].items():
            print(f"  {cls:12s} {cnt:>5}")
    
    print(f"\n✓ Training data ready: {TRAINING_DATA_YAML}")
    return manifest


In [ ]:
# Uncomment to run:
run_task2()

## Common libraries + helper functions

In [ ]:
%matplotlib inline
import cv2
import math
from pathlib import Path
from matplotlib import pyplot as plt
from ultralytics import YOLO

def display_image_subplots(image_filenames, image_dir, num_images, num_cols):
    if num_cols <= 0:
        raise ValueError("num_cols must be greater than 0")
    if num_images <= 0:
        raise ValueError("num_images must be greater than 0")
    if not image_filenames:
        raise ValueError("image_filenames cannot be empty")

    image_dir = Path(image_dir)
    selected_files = image_filenames[: min(num_images, len(image_filenames))]
    num_rows = math.ceil(len(selected_files) / num_cols)

    fig, axes = plt.subplots(num_rows, num_cols, figsize=(15 * num_cols, 10 * num_rows))
    axes = axes.flatten() if hasattr(axes, "flatten") else [axes]

    for i, filename in enumerate(selected_files):
        img_path = image_dir / filename
        img = cv2.imread(str(img_path))

        if img is None:
            axes[i].text(0.5, 0.5, f"Missing:\n{filename}", ha="center", va="center")
            axes[i].set_title(filename)
            axes[i].axis("off")
            continue

        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[i].imshow(img_rgb)
        axes[i].set_title(filename)
        axes[i].axis("off")

    # Hide any unused subplot slots
    for j in range(len(selected_files), len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    plt.show()

# CONSTANT
GRAPH_FILENAMES = [
    "results.png",
    "confusion_matrix.png",
    "BoxF1_curve.png",
    "BoxR_curve.png",
    "BoxP_curve.png"
]

DATASET_DIR = str(CURATED_YOLO_DIR)

## Task 3

### Train: default settings

In [ ]:

# 1) load pretrained YOLOv8 model
model = YOLO("yolov8n.pt")

# 2) train using your dataset yaml
model.train(
    data=str(TRAINING_DATA_YAML),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0  # use "cpu" if no GPU
)

In [ ]:
# 3) display training graphs
display_image_subplots(
    image_filenames=GRAPH_FILENAMES,
    image_dir="runs/detect/train",
    num_images=5,
    num_cols=1,
)

### Train 2: 
* increase epochs from 50 - 100
* image size increase
* patience

In [ ]:
from ultralytics import YOLO

# 1) load pretrained YOLOv8 model
model = YOLO("yolov8n.pt")

# 2) train using your dataset yaml (with workers=0 to fix Windows multiprocessing issue)
model.train(
    data=str(TRAINING_DATA_YAML),
    epochs=100,
    imgsz=800,
    batch=16,
    device=0,  # use "cpu" if no GPU
    workers=0,  # Fix for Windows: disable multiprocessing
    patience=20  # Early stopping if no improvement
)

In [ ]:

display_image_subplots(
    image_filenames=GRAPH_FILENAMES,
    image_dir="runs/detect/train2",
    num_images=5,
    num_cols=1,
)


## Train 3: 
* add data augmentations

In [ ]:
from ultralytics import YOLO

# 1) load pretrained YOLOv8 model
model = YOLO("yolov8n.pt")

# 2) train using your dataset yaml (with workers=0 to fix Windows multiprocessing issue)
model.train(
    data=str(TRAINING_DATA_YAML),
    epochs=100,
    imgsz=800,
    batch=16,
    device=0, 
    workers=0,  
    patience=20, 

    # Data augmentation parameters
    mosaic=1.0,
    mixup=0.2,
    scale=0.5
)

In [ ]:
display_image_subplots(
    image_filenames=GRAPH_FILENAMES,
    image_dir="runs/detect/train3",
    num_images=5,
    num_cols=1,
)


## Train 4: 
* freeze first 5 layers

In [ ]:
from ultralytics import YOLO

# 1) load pretrained YOLOv8 model
model = YOLO("yolov8n.pt")

# 2) train using your dataset yaml (with workers=0 to fix Windows multiprocessing issue)
model.train(
    data=str(TRAINING_DATA_YAML),
    epochs=100,
    imgsz=800,
    batch=16,
    device=0, 
    workers=0,  
    patience=20, 
    freeze=5,  # Freeze first 5 layers

    # Data augmentation parameters
    mosaic=1.0,
    mixup=0.2,
    scale=0.5
)

In [ ]:
display_image_subplots(
    image_filenames=GRAPH_FILENAMES,
    image_dir="runs/detect/train4",
    num_images=5,
    num_cols=1,
)

## Train 5: YOLOv8s (Upgraded Model)

**Key changes from previous experiments:**
- **Model**: YOLOv8s (11.2M params) instead of YOLOv8n (3.2M params) - 3.5x more capacity
- **Cosine LR**: Smoother learning rate decay for better convergence
- **Dropout**: Light regularization (0.1) to reduce overfitting
- **Extended training**: 100 epochs with patience=30

**Justification:**
- Previous experiments with YOLOv8n plateaued at mAP50 ~0.41
- Larger model can learn more complex cat-object interactions
- Cosine LR often outperforms linear decay on small datasets
- Dropout helps with the train/val loss gap observed in previous runs

In [ ]:
from ultralytics import YOLO

# Load YOLOv8s (small) - 11.2M parameters vs YOLOv8n's 3.2M
model = YOLO("yolov8s.pt")

# Train with optimized hyperparameters
results = model.train(
    data=str(TRAINING_DATA_YAML),
    epochs=100,
    imgsz=640,
    batch=16,
    patience=30,          # Extended patience for larger model
    seed=42,              # Explicit seed for reproducibility
    cos_lr=True,          # Cosine annealing learning rate schedule
    dropout=0.1,          # Light regularization
    mixup=0.1,            # Reduced from 0.2 (didn't help much)
    close_mosaic=15,      # Extended mosaic phase
    project="runs/detect",
    name="train5_yolov8s",
    plots=True,
    verbose=True,
)

In [ ]:
# Display training results for YOLOv8s
display_image_subplots(
    image_filenames=GRAPH_FILENAMES,
    image_dir="runs/detect/train5_yolov8s",
    num_images=5,
    num_cols=2
)